In [1]:
import pandas as pd
import math

In [2]:
sheet_lst = ['sales', 'total_store_list', 'treatment_control_pairs', 'step_1']
sheet_dic = {}

for sheet in sheet_lst:
    sheet_dic[sheet] = pd.read_csv(f'materials/3-17_ab-test-walkthrough/sales-raw-{sheet}.csv')

In [3]:
tmp_df = sheet_dic['step_1']
tmp_df = tmp_df[tmp_df['Period']=='Comp'].groupby(['Store'], as_index=False)['Sum_Gross_Sales'].mean()
tmp_df = pd.DataFrame(tmp_df)
tmp_df.columns = ['Store', 'Avg Gross Sales for Entire Comp Period']

step1_df = pd.merge(sheet_dic['sales'], tmp_df, how='left', on='Store')
del tmp_df

step1_df['Gross Sales Growth'] = step1_df.apply(lambda row: round((row['Sum_Gross_Sales']-row['Avg Gross Sales for Entire Comp Period'])/row['Avg Gross Sales for Entire Comp Period'], 3), axis=1)
#step1_df['Gross Sales Growth'] = step1_df.apply(lambda row: (row['Sum_Gross_Sales']-row['Avg Gross Sales for Entire Comp Period'])/row['Avg Gross Sales for Entire Comp Period'], axis=1)
step1_df.head(20)

,Region,Store,Week,Week_Start,Week_End,Period,Sum_Gross_Sales,Avg Gross Sales for Entire Comp Period,Gross Sales Growth
0,Midwest,store_012,49,2014/1/9,2014/1/15,Comp,1682.54,1796.796,-0.064
1,Midwest,store_012,50,2014/1/16,2014/1/22,Comp,2387.10,1796.796,0.329
2,Midwest,store_012,51,2014/1/23,2014/1/29,Comp,1847.11,1796.796,0.028
3,Midwest,store_012,52,2014/1/30,2014/2/5,Comp,1192.87,1796.796,-0.336
4,Midwest,store_012,53,2014/2/6,2014/2/12,Comp,2996.67,1796.796,0.668
5,Midwest,store_012,54,2014/2/13,2014/2/19,Comp,1800.50,1796.796,0.002
6,Midwest,store_012,55,2014/2/20,2014/2/26,Comp,1087.62,1796.796,-0.395
7,Midwest,store_012,56,2014/2/27,2014/3/5,Comp,1655.80,1796.796,-0.078
8,Midwest,store_012,57,2014/3/6,2014/3/12,Comp,2315.13,1796.796,0.288
9,Midwest,store_012,58,2014/3/13,2014/3/19,Comp,1002.62,1796.796,-0.442


In [4]:
step1_subdf = step1_df[['Store', 'Avg Gross Sales for Entire Comp Period']].drop_duplicates()
step1_subdf

,Store,Avg Gross Sales for Entire Comp Period
0,store_012,1796.796000
20,store_015,1890.905000
40,store_024,1401.447000
60,store_027,1155.485556
77,store_029,2013.836000
...,...,...
1670,store_788,2915.016000
1690,store_796,1884.956000
1710,store_798,2291.301000
1730,store_802,2774.967000


In [5]:
tmp_df = step1_df.groupby(['Store', 'Period'], as_index=False)['Gross Sales Growth'].mean()
step2_df = tmp_df.pivot(index='Store', columns='Period')['Gross Sales Growth'].reset_index()

del tmp_df
step2_df.head(20)

Period,Store,Comp,Test
0,store_012,-5.551115e-18,0.098800
1,store_015,-1.110223e-17,-0.542900
2,store_024,-1.000000e-04,0.200100
3,store_027,-1.111111e-04,-0.211625
4,store_029,1.000000e-04,-0.187700
5,store_033,-2.000000e-04,-0.503900
6,store_150,0.000000e+00,-0.153600
7,store_251,-1.110223e-17,-0.032222
8,store_299,-1.000000e-04,-0.080800
9,store_323,-1.387779e-18,-0.115700


In [6]:
def pair(store, treatment_pairing, control):
    if control == 'Control':
        return f'{store} <-> {treatment_pairing}'
    else:
        return store

tmp_df = sheet_dic['treatment_control_pairs'][['Controls', 'Treatments', 'Region']]
tmp_df.columns = ['Store', 'Treatment-Pairing', 'Region']

tmp_df = pd.merge(sheet_dic['total_store_list'], tmp_df, how='left', on=['Store', 'Region'])
tmp_df['Control-Treatment Pairing'] = tmp_df.apply(lambda row: pair(row['Store'], row['Treatment-Pairing'], row['Type']), axis=1)

step3_df = tmp_df[['Store', 'Treatment-Pairing', 'Region', 'Type', 'Control-Treatment Pairing']]
#step3_df.columns = ['store_no', 'Treatment-Pairing', 'Region', 'Type', 'Control-Treatment Pairing']

del tmp_df
step3_df.head(20)

,Store,Treatment-Pairing,Region,Type,Control-Treatment Pairing
0,store_015,NaN,Midwest,Treatment,store_015
1,store_024,NaN,Midwest,Treatment,store_024
2,store_027,NaN,Midwest,Treatment,store_027
3,store_695,NaN,Midwest,Treatment,store_695
4,store_696,NaN,Midwest,Treatment,store_696
5,store_707,NaN,Midwest,Treatment,store_707
6,store_719,NaN,Midwest,Treatment,store_719
7,store_721,NaN,Midwest,Treatment,store_721
8,store_724,NaN,Midwest,Treatment,store_724
9,store_727,NaN,Midwest,Treatment,store_727


In [7]:
step4_df = pd.merge(step3_df, step2_df, how='left')
step4_df.columns = ['Store', 'Treatment-Pairing', 'Region', 'Type', 'Control-Treatment Pairing', 'Avg Growth Comparative Period', 'Avg Growth Test Period']

step4_df.to_csv('step4_df.csv')

step4_df

,Store,Treatment-Pairing,Region,Type,Control-Treatment Pairing,Avg Growth Comparative Period,Avg Growth Test Period
0,store_015,NaN,Midwest,Treatment,store_015,-1.110223e-17,-0.542900
1,store_024,NaN,Midwest,Treatment,store_024,-1.000000e-04,0.200100
2,store_027,NaN,Midwest,Treatment,store_027,-1.111111e-04,-0.211625
3,store_695,NaN,Midwest,Treatment,store_695,2.000000e-04,-0.406800
4,store_696,NaN,Midwest,Treatment,store_696,-9.367507e-18,-0.374500
...,...,...,...,...,...,...,...
78,store_636,store_769,Midwest,Control,store_636 <-> store_769,1.000000e-04,-0.262800
79,store_594,store_596,West,Control,store_594 <-> store_596,1.110223e-17,0.042200
80,store_526,store_596,West,Control,store_526 <-> store_596,1.000000e-04,-0.331100
81,store_478,store_601,West,Control,store_478 <-> store_601,-2.775558e-18,-0.014000


In [8]:
step5_df = step4_df.copy()
step5_df['Growth_Difference'] = step5_df.apply(lambda row: row['Avg Growth Test Period'] - row['Avg Growth Comparative Period'], axis=1)
step5_df

,Store,Treatment-Pairing,Region,Type,Control-Treatment Pairing,Avg Growth Comparative Period,Avg Growth Test Period,Growth_Difference
0,store_015,NaN,Midwest,Treatment,store_015,-1.110223e-17,-0.542900,-0.542900
1,store_024,NaN,Midwest,Treatment,store_024,-1.000000e-04,0.200100,0.200200
2,store_027,NaN,Midwest,Treatment,store_027,-1.111111e-04,-0.211625,-0.211514
3,store_695,NaN,Midwest,Treatment,store_695,2.000000e-04,-0.406800,-0.407000
4,store_696,NaN,Midwest,Treatment,store_696,-9.367507e-18,-0.374500,-0.374500
...,...,...,...,...,...,...,...,...
78,store_636,store_769,Midwest,Control,store_636 <-> store_769,1.000000e-04,-0.262800,-0.262900
79,store_594,store_596,West,Control,store_594 <-> store_596,1.110223e-17,0.042200,0.042200
80,store_526,store_596,West,Control,store_526 <-> store_596,1.000000e-04,-0.331100,-0.331200
81,store_478,store_601,West,Control,store_478 <-> store_601,-2.775558e-18,-0.014000,-0.014000


In [9]:
from scipy import stats

In [10]:
arr_1 = step5_df[step5_df['Type']=='Treatment']['Growth_Difference']
arr_2 = step5_df[step5_df['Type']=='Control']['Growth_Difference']
ttest_results = stats.ttest_ind(arr_1, arr_2, equal_var=False)
ttest_results

Ttest_indResult(statistic=-1.1924271050832889, pvalue=0.24692900792352512)

In [11]:
f'{round(1 - ttest_results.pvalue, 3)*100}%'

'75.3%'

In [12]:
tmp_df = sheet_dic['total_store_list']
tmp_df = tmp_df[tmp_df['Type'] == 'Treatment'][['Store']]
tmp_df.columns = ['Treatments']

tmp_df2 = step5_df[['Store', 'Treatment-Pairing', 'Region', 'Growth_Difference']]
tmp_df2.columns = ['Controls', 'Treatments', 'Region', 'Difference_GrossSales_Control']

step6_df = pd.merge(tmp_df, tmp_df2, how='left')

tmp_df.columns = ['Controls']
tmp_df3 = pd.merge(tmp_df, tmp_df2, how='left')
tmp_df3 = tmp_df3[['Controls', 'Difference_GrossSales_Control']]
tmp_df3.columns = ['Treatments', 'Difference_GrossSales_Treatment']

step6_df = pd.merge(step6_df, tmp_df3, how='left')
step6_df = step6_df[['Controls', 'Treatments', 'Region', 'Difference_GrossSales_Control', 'Difference_GrossSales_Treatment']]

In [13]:
step6_df['Lift'] = step6_df.apply(lambda row: (row['Difference_GrossSales_Treatment']-row['Difference_GrossSales_Control'])/(1+row['Difference_GrossSales_Control']), axis=1)

del tmp_df
del tmp_df2
del tmp_df3

In [14]:
tmp_df = step1_subdf.copy()
tmp_df.columns = ['Treatments', 'Avg Sales from Comp Period']
step6_df = pd.merge(step6_df, tmp_df, how='left', on='Treatments').drop_duplicates().reset_index()
step6_df['Expected Sales Impact'] = step6_df.apply(lambda row: row['Lift']*row['Avg Sales from Comp Period'], axis=1)

del tmp_df

step6_df

,index,Controls,Treatments,Region,Difference_GrossSales_Control,Difference_GrossSales_Treatment,Lift,Avg Sales from Comp Period,Expected Sales Impact
0,0,store_687,store_015,Midwest,-0.059200,-0.542900,-0.514137,1890.905000,-972.184044
1,1,store_728,store_015,Midwest,0.174100,-0.542900,-0.610681,1890.905000,-1154.738851
2,2,store_597,store_024,Midwest,-0.515400,0.200200,1.476682,1401.447000,2069.491278
3,3,store_776,store_024,Midwest,-0.130300,0.200200,0.380016,1401.447000,532.572420
4,10,store_597,store_027,Midwest,-0.515400,-0.211514,0.627086,1155.485556,724.589377
5,11,store_776,store_027,Midwest,-0.130300,-0.211514,-0.093381,1155.485556,-107.900972
6,18,store_654,store_695,Midwest,-0.351700,-0.407000,-0.085300,1903.757000,-162.390501
7,19,store_550,store_695,Midwest,-0.245300,-0.407000,-0.214257,1903.757000,-407.893874
8,20,store_776,store_696,Midwest,-0.130300,-0.374500,-0.280786,1797.294000,-504.655852
9,22,store_788,store_696,Midwest,0.109300,-0.374500,-0.436131,1797.294000,-783.855438


In [15]:
avg_lift = round(step6_df['Lift'].mean(), 3)*100
f'{avg_lift}%'

'-1.6%'

In [16]:
avg_expected_sales_impact = round(step6_df['Expected Sales Impact'].mean(), 1)
avg_expected_sales_impact

-122.4